# ODA-LAB extract SET-3

**Runtime → Run all.** Third window.

Leftover sunset/pixel packs + preview tar + four conversation tars from `02_REMAINING_SUNSETS`.
Not repeating SET-1 or SET-2 IDs. Not the 682 MB GGUF. Not Google Docs.
CAP 500 MB. Dest `12_ODA-LAB-NOTEBOOKS/extracts/SET3-<stamp>/`.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, subprocess
print('=== SET-3 mount ===')
drive.mount('/content/drive', force_remount=False)
ROOT = Path('/content/drive/MyDrive')
print('mounted', ROOT.exists())
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
print('gdown ok')


In [ ]:
import gdown, zipfile, tarfile, time
CAP = 500 * 1024 * 1024
PACKS = [
  [
    "sunset_full_2026-09-10.zip",
    "14vU0nA6IwQGbzXXb7G6ZALBsnYkD0Fhg",
    35981,
    "zip"
  ],
  [
    "2026-09-10_SUNSET_TEXT_PACKAGES.zip",
    "1h68ZxmzfpGlxykGE1orYmZZyFm2Vxlms",
    73695,
    "zip"
  ],
  [
    "sunset_pixels_other_2026-09-10.zip",
    "1o2Ax6cuFTwu9d9XujuoUlBGtozH5s0ef",
    7801401,
    "zip"
  ],
  [
    "sunset_pixels_imagine_rendered_2026-09-10.tar.gz",
    "1NVMHd5mUgXntJcTgC5TuAUC9XtBTlsTD",
    12000000,
    "tar.gz"
  ],
  [
    "sunset_full_archive_2026-09-10.zip",
    "17Fhlp5gm0EzPrI7j6bnzHTzzGtVH5ZG0",
    6258965,
    "zip"
  ],
  [
    "2026-09-10_SUNSET_PIXELS_rendered_and_attachments.zip",
    "1XlvwDgoNCsFGsPTQAidbBrlMs0Qlre6W",
    64300000,
    "zip"
  ],
  [
    "sunset_pixels_2026-09-10.tar.gz",
    "19VZ0bGNYMUw5Wnh2GusMhn08I-fSg-QX",
    82300000,
    "tar.gz"
  ],
  [
    "preview-only-message.tar.gz",
    "1fs7ape_GBlnpiZY3CiiXo7yVMbRSqghY",
    0,
    "tar.gz"
  ],
  [
    "sunset-003-full-archive.tar.gz",
    "1e7yBIy4sH4zi10i2kh2_Ik1ikETk5PNH",
    15000000,
    "tar.gz"
  ],
  [
    "SEQ-Crepax-pixel.tar.gz",
    "10yxkVArb7AwQ78WixYUNWUU6baPeW9IS",
    0,
    "tar.gz"
  ],
  [
    "full-pixels-003-fat.tar.gz",
    "1OEd7nBVPkZPmdvy-UUE2of-2tMOTKvBh",
    101000000,
    "tar.gz"
  ],
  [
    "imgpipe-memory-BIOS-004.tar.gz",
    "137uVQqiWHjcg9ezudMNWsUg2TvArz2bo",
    89000000,
    "tar.gz"
  ]
]
shelf = None
nwalk = 0
for dirpath, dirnames, filenames in os.walk(ROOT):
    nwalk += 1
    if '12_ODA-LAB-NOTEBOOKS' in dirnames:
        shelf = Path(dirpath) / '12_ODA-LAB-NOTEBOOKS'
        break
    if nwalk > 4000:
        break
if shelf is None:
    shelf = ROOT / '12_ODA-LAB-NOTEBOOKS_LOCAL'
    shelf.mkdir(exist_ok=True)
dest_root = shelf / 'extracts' / ('SET3-' + time.strftime('%Y%m%d-%H%M'))
dest_root.mkdir(parents=True, exist_ok=True)
print('dest', dest_root)

def find_name(name):
    n = 0
    for dirpath, dirnames, filenames in os.walk(ROOT):
        n += len(filenames)
        if name in filenames:
            return Path(dirpath) / name
        if n > 12000:
            break
    return None

lines = ['# ODA LAB extract SET-3', 'dest=' + str(dest_root), '']
for name, fid, claimed, kind in PACKS:
    print('\n===', kind, name, fid)
    src = find_name(name)
    if src is None:
        local = Path('/content') / name
        print('gdown', fid)
        try:
            gdown.download(id=fid, output=str(local), quiet=False)
            src = local if local.is_file() else None
        except Exception as e:
            print('gdown fail', e)
            src = None
    if src is None or not src.is_file():
        lines.append('- FAIL missing ' + name + ' ' + fid)
        continue
    sz = src.stat().st_size
    print('src', src, 'size', sz)
    if sz > CAP:
        lines.append('- SKIP cap %s %s' % (name, sz))
        continue
    out = dest_root / Path(name).name.split('.')[0]
    if out.exists() and any(out.iterdir()):
        cnt = sum(1 for _ in out.rglob('*') if _.is_file())
        lines.append('- EXISTS %s files=%s' % (out, cnt))
        continue
    out.mkdir(parents=True, exist_ok=True)
    try:
        if kind == 'zip':
            with zipfile.ZipFile(src) as z:
                z.extractall(out)
                nlist = z.namelist()[:6]
        else:
            with tarfile.open(src, 'r:*') as t:
                t.extractall(out)
                nlist = t.getnames()[:6]
        cnt = sum(1 for _ in out.rglob('*') if _.is_file())
        lines.append('- OK %s bytes=%s files=%s first=%s' % (name, sz, cnt, nlist))
        print('OK', cnt)
    except Exception as e:
        lines.append('- FAIL extract %s %s' % (name, e))
        print('FAIL', e)
receipt = dest_root / 'EXTRACT.RECEIPT.md'
receipt.write_text('\n'.join(str(x) for x in lines))
print('\nWROTE', receipt)
print('\n'.join(str(x) for x in lines))
